In [ ]:
import os
import math
from datasets import load_dataset,Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, PeftModel
import torch
import numpy as np

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-Math-1.5B-Instruct"
DATA_PATH = "grade5_6_math_mcq_7000.json"
OUTPUT_DIR = "./qgen_model"

MAX_LENGTH = 512
BATCH_SIZE = 2
GRAD_ACCUM = 4
EPOCHS = 4
LR = 2e-5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PER_DEVICE_BATCH_SIZE = 4
SEED = 42
DATALOADER_NUM_WORKERS = 0

In [ ]:
import json
import random

def build_prompt(example):
    seed_hint = random.randint(1000, 9999)
    return (
        "### INSTRUCTION\n"
        "Generate ONE UNIQUE math MCQ.\n"
        "The question MUST be different from previous ones.\n"
        "Understand the mathematcal relationship and verify the answer before giving the final answer.\n"
        f"Use a different scenario. Seed: {seed_hint}\n\n"
        "### METADATA\n"
        f"Topic: {example['topic']}\n"
        f"Difficulty: {example['difficulty']}\n\n"
        "### OUTPUT (JSON ONLY)\n"
    )

def build_completion(example):
    completion_obj = {
        "question": example["question"],
        "choices": {
            "A": example["A"],
            "B": example["B"],
            "C": example["C"],
            "D": example["D"],
        },
        "reasoning": example.get("reasoning", ""),
        "verification": example.get("verification", ""),
        "correct_answer": example["correct"]
    }

    return json.dumps(completion_obj, ensure_ascii=False)

In [ ]:
def build_full_prompt(example):
    prompt = build_prompt(example)
    completion_lines = build_completion(example)

    completion = "\n" + completion_lines

    example["prompt"] = prompt
    example["completion"] = " " + completion  # keep your leading space
    return example

In [ ]:
from networkx import display
import pandas as pd

df = pd.read_csv('grade5_6_math_mcq_7000.csv')
display(df.head())

,topic,difficulty,question,A,B,C,D,correct,reasoning,verification
0,arithmetic,easy,What is 50 + 17?,67.0,62.0,48.0,64.0,A,Step 1: Identify the mathematical relationship...,The computed result is 67.0. Option A equals 6...
1,arithmetic,easy,What is 44 + 15?,66.0,76.0,59.0,41.0,C,Step 1: Identify the mathematical relationship...,The computed result is 59.0. Option C equals 5...
2,arithmetic,easy,What is 24 + 42?,81.0,66.0,47.0,84.0,B,Step 1: Identify the mathematical relationship...,The computed result is 66.0. Option B equals 6...
3,arithmetic,easy,What is 24 + 38?,42.0,62.0,79.0,59.0,B,Step 1: Identify the mathematical relationship...,The computed result is 62.0. Option B equals 6...
4,arithmetic,easy,What is 31 + 27?,47.0,51.0,58.0,59.0,C,Step 1: Identify the mathematical relationship...,The computed result is 58.0. Option C equals 5...


In [ ]:
display(build_full_prompt(df.iloc[0]))

/tmp/ipython-input-1510419324.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  example["prompt"] = prompt
/tmp/ipython-input-1510419324.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  example["prompt"] = prompt
/tmp/ipython-input-1510419324.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  example["completion"] = " " + completion  # keep your leading space
/tmp/ipython-input-1510419324.py:8: SettingWithCopyWarning: 
A value is

,0
topic,arithmetic
difficulty,easy
question,What is 50 + 17?
A,67.0
B,62.0
C,48.0
D,64.0
correct,A
reasoning,Step 1: Identify the mathematical relationship...
verification,The computed result is 67.0. Option A equals 6...


In [ ]:
df_new = Dataset.from_pandas(df)

In [ ]:
df_new = df_new.map(build_full_prompt)

Map:   0%|          | 0/6990 [00:00<?, ? examples/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)


def tokenize_fn(batch):
    input_ids_batch = []
    attention_mask_batch = []
    labels_batch = []

    for p, c in zip(batch["prompt"], batch["completion"]):
        # Tokenize the combined text.
        # We set padding=False to return raw lists of integers.
        # We will pad dynamically in the collator.
        tokenized_full_text = tokenizer(
            p + c,
            max_length=MAX_LENGTH,
            truncation=True,
            padding=False,
            return_attention_mask=True,
            add_special_tokens=True
        )

        input_ids = tokenized_full_text["input_ids"]
        attention_mask = tokenized_full_text["attention_mask"]

        # Create labels. Mask the prompt part with -100.
        # Re-tokenize prompt to get its length.
        prompt_only_tokens = tokenizer(p, add_special_tokens=True, truncation=True, max_length=MAX_LENGTH)["input_ids"]
        effective_prompt_len = len(prompt_only_tokens)

        # Safety check: ensure prompt length isn't longer than full input
        effective_prompt_len = min(effective_prompt_len, len(input_ids))

        labels = [-100] * effective_prompt_len + input_ids[effective_prompt_len:]

        input_ids_batch.append(input_ids)
        attention_mask_batch.append(attention_mask)
        labels_batch.append(labels)

    return {
        "input_ids": input_ids_batch,
        "attention_mask": attention_mask_batch,
        "labels": labels_batch
    }

# Re-process the dataset
if 'df' in globals():
    df_new = Dataset.from_pandas(df)
    # Ensure build_full_prompt is defined or imported, assuming it exists in the notebook context
    if 'build_full_prompt' in globals(): # Corrected typo: buid_full_prompt -> build_full_prompt
        df_new = df_new.map(build_full_prompt) # Corrected typo: buid_full_prompt -> build_full_prompt
    df_new = df_new.map(tokenize_fn, batched=True, remove_columns=df_new.column_names)
    df_new.set_format(type="torch")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/6990 [00:00<?, ? examples/s]

Map:   0%|          | 0/6990 [00:00<?, ? examples/s]

In [ ]:
# Split the dataset into train and test
# We use the df_new dataset which has been tokenized
split_datasets = df_new.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split_datasets["train"]
eval_dataset = split_datasets["test"]

print(f"Train set size: {len(train_dataset)}")
print(f"Test set size: {len(eval_dataset)}")

Train set size: 6291
Test set size: 699


In [ ]:
from transformers import DataCollatorForSeq2Seq

# DataCollatorForSeq2Seq handles dynamic padding for input_ids and labels
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8
)

In [ ]:
!pip install bitsandbytes peft accelerate

In [ ]:
from peft import prepare_model_for_kbit_training

from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto"
)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(base, lora)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    fp16=torch.cuda.is_available(),
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=1,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    load_best_model_at_end=True,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

In [ ]:
import gc
import torch

# Clear cache before training
torch.cuda.empty_cache()
gc.collect()

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akashgeethanjana320 (akashgeethanjana320-individual) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:740: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss
1,0.191700,0.165651
2,0.152600,0.151480
3,0.147000,0.147762
4,0.143000,0.146607


TrainOutput(global_step=1576, training_loss=0.23607911373758075, metrics={'train_runtime': 1996.1045, 'train_samples_per_second': 12.607, 'train_steps_per_second': 0.79, 'total_flos': 4.264142313150874e+16, 'train_loss': 0.23607911373758075, 'epoch': 4.0})

In [ ]:
DRIVE_MODEL_PATH = "/content/drive/MyDrive/Question_Pretrained"

# Save the model (LoRA adapters) and tokenizer to Google Drive
model.save_pretrained(DRIVE_MODEL_PATH)
tokenizer.save_pretrained(DRIVE_MODEL_PATH)

('/content/drive/MyDrive/Question_Pretrained/tokenizer_config.json',
 '/content/drive/MyDrive/Question_Pretrained/special_tokens_map.json',
 '/content/drive/MyDrive/Question_Pretrained/chat_template.jinja',
 '/content/drive/MyDrive/Question_Pretrained/vocab.json',
 '/content/drive/MyDrive/Question_Pretrained/merges.txt',
 '/content/drive/MyDrive/Question_Pretrained/added_tokens.json',
 '/content/drive/MyDrive/Question_Pretrained/tokenizer.json')

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:

torch.cuda.empty_cache()
import gc
gc.collect()

# 2. Load Tokenizer & Ensure Special Tokens exist
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

# 3. Load Base Model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    device_map="auto"
)


# 5. Load Adapters
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/Question_Pretrained")
model.eval()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear4bit(
       

In [ ]:
def generate_question(topic, difficulty):
    seed_hint = random.randint(1000, 9999)
    prompt = (
        "### INSTRUCTION\n"
        "Generate ONE UNIQUE math MCQ.\n"
        "The question MUST be different from previous ones.\n"
        "Understand the mathematcal relationship and verify the answer before giving the final answer.\n"
        f"Use a different scenario. Seed: {seed_hint}\n\n"
        "### METADATA\n"
        f"Topic: {topic}\n"
        f"Difficulty: {difficulty}\n\n"
        "### OUTPUT (JSON ONLY)\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)


    output = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2
    )

    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text


In [ ]:
sample = generate_question("probability", "easy")
print(sample)

### INSTRUCTION
Generate ONE UNIQUE math MCQ.
The question MUST be different from previous ones.
Understand the mathematcal relationship and verify the answer before giving the final answer.
Use a different scenario. Seed: 9935

### METADATA
Topic: probability
Difficulty: easy

### OUTPUT (JSON ONLY)
 
{"question": "What comes after 1?", "choices": {"A": -0.0, "B": 2.0, "C": 4.0, "D": 6.0}, "reasoning": "Step 1: Identify the mathematical relationship in the question. Step 2: Apply the correct arithmetic operation to solve it. Step 3: The computed result equals 2.0.", "verification": "The computed result is 2.0. Option B equals 2.0, so it is correct.", "correct_answer": "B"}]
``` To generate an7o unique mathMCQ with the given properties:
1. **Question creating**: Ensure each generated question is distinct


In [ ]:
import os
import shutil
# from google.colab import drive

# Mount Google Drive
# drive.mount('/content/drive')
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

# Define local output directory and Google Drive destination paths
OUTPUT_DIR = "./qgen_model"
drive_base_path = "/content/drive/MyDrive/Question_Pretrained"
drive_destination_folder = os.path.join(drive_base_path, os.path.basename(OUTPUT_DIR))

# Create the base destination directory in Google Drive if it doesn't exist
os.makedirs(drive_base_path, exist_ok=True)
# Create the specific model directory (e.g., 'qgen_model') in Google Drive if it doesn't exist
os.makedirs(drive_destination_folder, exist_ok=True)

print(f"Moving contents from '{OUTPUT_DIR}' to '{drive_destination_folder}'")

# List all items (files and directories) in the local OUTPUT_DIR
items_to_move = os.listdir(OUTPUT_DIR)

# Exclude specific items as requested by the user
excluded_items = ["checkpoint-788", "runs"]

moved_count = 0
for item in items_to_move:
    if item not in excluded_items:
        source_path = os.path.join(OUTPUT_DIR, item)
        destination_path = os.path.join(drive_destination_folder, item)
        print(f"Moving '{source_path}' to '{destination_path}'")
        try:
            shutil.move(source_path, destination_path)
            moved_count += 1
        except Exception as e:
            print(f"Error moving {source_path}: {e}")

print(f"Successfully moved {moved_count} items to '{drive_destination_folder}'")

# Verify contents in Drive
print(f'Contents of {drive_destination_folder}:')
!ls -F '{drive_destination_folder}'

# Clean up the local OUTPUT_DIR if it's empty or only contains excluded items
# Check if OUTPUT_DIR is empty after moving
if not os.listdir(OUTPUT_DIR):
    print(f"Local directory '{OUTPUT_DIR}' is empty. Removing it.")
    shutil.rmtree(OUTPUT_DIR)
else:
    print(f"Local directory '{OUTPUT_DIR}' still contains items (possibly excluded ones): {os.listdir(OUTPUT_DIR)}")

print("Move operation completed.")

Mounted at /content/drive
Moving contents from './qgen_model' to '/content/drive/MyDrive/Question_Pretrained/qgen_model'
Moving './qgen_model/adapter_model.safetensors' to '/content/drive/MyDrive/Question_Pretrained/qgen_model/adapter_model.safetensors'
Moving './qgen_model/special_tokens_map.json' to '/content/drive/MyDrive/Question_Pretrained/qgen_model/special_tokens_map.json'
Moving './qgen_model/added_tokens.json' to '/content/drive/MyDrive/Question_Pretrained/qgen_model/added_tokens.json'
Moving './qgen_model/adapter_config.json' to '/content/drive/MyDrive/Question_Pretrained/qgen_model/adapter_config.json'
Moving './qgen_model/tokenizer_config.json' to '/content/drive/MyDrive/Question_Pretrained/qgen_model/tokenizer_config.json'
Moving './qgen_model/training_args.bin' to '/content/drive/MyDrive/Question_Pretrained/qgen_model/training_args.bin'
Moving './qgen_model/merges.txt' to '/content/drive/MyDrive/Question_Pretrained/qgen_model/merges.txt'
Moving './qgen_model/chat_template